# Lab: validate container and CI definitions as data
This notebook never invokes Docker, subprocess, shell, network, or filesystem writes.


In [ ]:
import re
print('Python environment ready; definitions will be inspected as strings')


## Objectives
Find mutable inputs, root runtime, secret leaks, missing stages, and a pipeline gate that promotes too early.


## Baseline reproduction — Predict 1
A Dockerfile with `USER root` and `FROM python:latest` is repeatable and least-privilege safe. Predict before running.


In [ ]:
baseline = 'FROM python:latest\nWORKDIR /app\nCOPY . .\nUSER root\nCMD ["python","app.py"]'
def docker_policy(text):
    return {'pinned': ':latest' not in text and '@sha256:' in text, 'non_root': 'USER app' in text, 'workdir': 'WORKDIR ' in text, 'has_cmd': 'CMD ' in text}
bad = docker_policy(baseline)
assert bad['pinned'] is False and bad['non_root'] is False
print(bad)


**Pre-edit hypothesis:** a pinned base reference and an explicit non-root user will satisfy two policy checks without changing application behavior.


## Predict 2
If a pipeline lists tests after deploy, can a failing test block promotion? No; the ordering is unsafe.


In [ ]:
pipeline = ['checkout@v1','build','deploy','test','smoke']
def gate_order(stages):
    required = ['checkout','test','security','build','health','promote']
    positions = {name: next((i for i,s in enumerate(stages) if name in s), None) for name in required}
    return positions, positions['test'] is not None and positions['build'] is not None and positions['test'] < positions['build']
positions, safe = gate_order(pipeline)
assert safe is False
print(positions)


## Predict 3
If a secret-looking value is copied into an image definition, should policy accept it because it is synthetic? No: artifacts and logs must use runtime references, even in training.


In [ ]:
def contains_secret_literal(text):
    return bool(re.search(r'(PASSWORD|TOKEN|SECRET)\s*=', text, re.I))
definition = 'ENV DB_PASSWORD=synthetic-placeholder'
assert contains_secret_literal(definition)
print('Secret-like build literal detected')


## Guided TODO
Improve the definition in your notes: pinned digest, runtime user, workdir, and no secret literal. The next code cell is the executable reference solution.


In [ ]:
candidate = 'FROM python:3.12-slim@sha256:abc123\nWORKDIR /app\nUSER app\nCOPY --from=build /out /usr/local\nCMD [\"python\",\"app.py\"]'
policy = docker_policy(candidate)
assert policy == {'pinned': True, 'non_root': True, 'workdir': True, 'has_cmd': True}
assert not contains_secret_literal(candidate)


In [ ]:
# Reference pipeline: failing gates stop before build/promotion.
safe_pipeline = ['checkout@v1','format','unit-test','integration-test','security','build','inspect','health','promote']
pos, safe = gate_order(safe_pipeline)
assert safe and pos['security'] < pos['build'] and pos['health'] < pos['promote']
failing = {'unit-test': 'failed'}
assert failing['unit-test'] == 'failed' and 'build' not in failing
print('Pipeline ordering and failure-gate checks passed')


## Compose-like service validation
A service definition should make ports, dependency health, and synthetic runtime configuration visible without publishing a database or baking credentials.


In [ ]:
services = {'api': {'port': 8080, 'depends_on': 'db', 'secret_ref': 'DB_PASSWORD'}, 'db': {'internal': True}}
assert services['api']['depends_on'] == 'db'
assert services['api']['secret_ref'] and services['db']['internal']
print('Compose policy: dependency and secret reference are explicit')


## Intentionally weak AI-style definition
`latest`, root, `COPY . .`, environment dumps, and a deploy-before-test stage are plausible generated defaults. They may parse successfully but violate repeatability, least privilege, secret safety, or release gates.


## Independent challenge — attempt before checking
Write a release sequence with a compatible migration, candidate readiness, smoke test, promotion, and rollback. Put the order and one data-compatibility limitation in notes first, then compare with the executable check below.


In [ ]:
release = ['artifact','migration-compatible','start-candidate','readiness','smoke','promote']
assert release.index('readiness') < release.index('promote')
rollback_note = {'known_good':'digest-good','candidate':'digest-bad','data_limit':'migration must be backward compatible'}
assert rollback_note['data_limit']
print('Release and rollback note recorded')


## Exit questions
1. What does this notebook validate?
2. Why non-root and pinned inputs?
3. What remains unproved?

### Answers
1. Policy in text/data, not real image layers or a live deployment.
2. Smaller blast radius and repeatable artifact inputs.
3. Tool versions, registry behavior, actual health, migration locking, and deployment permissions.

## Evidence handoff
Save policy results, pipeline fail/pass ordering, secret scan, release sequence, rollback limitation, and AI diff critique.
